# 公益社団法人・公益財団法人の寄附金収入に関する実態調査 — 分析レポート

## 調査概要
- **出典**: 内閣府「公益法人の寄附金収入に関する実態調査」（令和元年度実施）
- **対象**: 公益社団法人・公益財団法人（全法人）
- **有効回答数**: 約3,987法人（回収率46.8%）、集計対象約6,155法人
- **データ期間**: 平成26年度〜平成30年度（2014〜2018年度）の寄附金データ
- **テーマ**: 公益法人の寄附金収入の実態・税額控除制度・募金活動

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import japanize_matplotlib
import openpyxl
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
COLORS = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63", "#9C27B0", "#00BCD4", "#FF5722", "#607D8B"]

DATA_DIR = "data06/公益社団法人及び公益財団法人の寄附金収入に関する実態調査/"
wb_s = openpyxl.load_workbook(DATA_DIR + "syukeidata_r1.xlsx", read_only=True, data_only=True)
wb_t = openpyxl.load_workbook(DATA_DIR + "集計表.xlsx", read_only=True, data_only=True)

def get_rows(wb, sheet):
    return list(wb[sheet].iter_rows(values_only=True))

print("データ読み込み完了")

## 1. 法人の基本属性

### 1.1 公益目的事業費用の規模分布

In [ ]:
rows = get_rows(wb_s, "図１")
labels = [str(v) for v in rows[7][3:] if v]
n_vals = [float(v) for v in rows[8][3:] if v is not None]
pct_vals = [float(v) for v in rows[9][3:] if v is not None]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(labels)), pct_vals, color=COLORS[:len(labels)], edgecolor="white", linewidth=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("構成比（%）")
ax.set_title("公益目的事業費用の規模分布（n=6,155）", fontsize=14, fontweight="bold")
for bar, v in zip(bars, pct_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nサンプル数: {int(n_vals[0]):,}")
print("\n--- 規模分布 ---")
for l, p in zip(labels, pct_vals):
    print(f"  {l}: {p:.1f}%")

**考察**: 公益目的事業費用20百万円未満の小規模法人が21.2%と最多を占める。一方、600百万円以上の大規模法人も16.7%存在し、規模の二極化が見られる。

### 1.2 職員数（常勤+非常勤）

In [ ]:
rows = get_rows(wb_s, "図３")
staff_labels = [str(v) for v in rows[7][3:] if v]
staff_pct = [float(v) for v in rows[9][3:] if v is not None]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(staff_labels)), staff_pct, color="#2196F3", alpha=0.85)
ax.set_xticks(range(len(staff_labels)))
ax.set_xticklabels(staff_labels, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("構成比（%）")
ax.set_title("職員数（常勤+非常勤）の分布（n=6,157）", fontsize=14, fontweight="bold")
for bar, v in zip(bars, staff_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 2. 寄附金収入の実態

### 2.1 寄附金収入金額の年次推移（全体）

In [ ]:
rows = get_rows(wb_s, "図７")
years = ["H26", "H27", "H28", "H29", "H30"]
zero_pct = []
for r_idx in [11, 13, 15, 17, 19]:
    zero_pct.append(float(rows[r_idx][4]))

# n行から平均金額取得
avg_amount = []
for r_idx in [10, 12, 14, 16, 18]:
    v = rows[r_idx][16]
    avg_amount.append(float(v) if isinstance(v, (int, float)) else 0)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(range(len(years)), zero_pct, color="#90CAF9", alpha=0.8, label="寄附金0円の法人割合（%）")
ax1.set_ylabel("0円法人の割合（%）", color="#1565C0")
ax1.set_ylim(50, 65)

ax2 = ax1.twinx()
ax2.plot(range(len(years)), [a/1000 for a in avg_amount], "o-", color="#E91E63",
         linewidth=2, markersize=8, label="平均金額（百万円）")
ax2.set_ylabel("平均金額（百万円）", color="#E91E63")

ax1.set_xticks(range(len(years)))
ax1.set_xticklabels(years)
ax1.set_title("寄附金収入金額の年次推移（全体）", fontsize=14, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.tight_layout()
plt.show()

print("\n--- 寄附金0円法人割合の推移 ---")
for y, z in zip(years, zero_pct):
    print(f"  {y}: {z:.1f}%")

**考察**: 寄附金収入が0円の法人は約55〜59%で推移。過半数の公益法人が寄附を受けていない。ただし平成30年度の0円割合は54.9%と改善傾向。

### 2.2 個人寄附 vs 法人寄附の金額帯分布（H30）

In [ ]:
rows_p = get_rows(wb_s, "図８")
rows_c = get_rows(wb_s, "図９")
cats = ["0円", "～50万", "50～100万", "100～200万", "200～300万",
        "300～400万", "400～600万", "600～800万", "800～1000万",
        "1000～5000万", "5000万～1億", "1億以上"]
pct_p = [float(v) for v in rows_p[19][4:16]]
pct_c = [float(v) for v in rows_c[19][4:16]]

x = np.arange(len(cats))
w = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, pct_p, w, label="個人寄附", color="#2196F3", alpha=0.85)
ax.bar(x + w/2, pct_c, w, label="法人寄附", color="#FF9800", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=40, ha="right", fontsize=8)
ax.set_ylabel("構成比（%）")
ax.set_title("寄附金額帯別分布：個人 vs 法人（平成30年度）", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"個人: 0円={pct_p[0]:.1f}%, 法人: 0円={pct_c[0]:.1f}%")
print(f"個人: 1億以上={pct_p[-1]:.1f}%, 法人: 1億以上={pct_c[-1]:.1f}%")

**考察**: 個人寄附は69.7%が0円、法人寄附は64.2%が0円。法人寄附の方が「寄附あり」の割合がやや高い。高額帯（1000万円以上）では法人寄附が個人を大幅に上回る。

## 3. 寄附金収入総額の推移（H20〜H24）

In [ ]:
rows = get_rows(wb_t, "３－１．寄附金収入額（公益法人全体）")
years_t = ["H20", "H21", "H22", "H23", "H24"]
total_b = [float(v)/1e6 for v in rows[5][2:7]]
kojin_b = [float(v)/1e6 for v in rows[6][2:7]]
hojin_b = [float(v)/1e6 for v in rows[7][2:7]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years_t, total_b, "o-", color="#1565C0", linewidth=2.5, markersize=8, label="寄附金収入計")
ax.plot(years_t, kojin_b, "s--", color="#E91E63", linewidth=2, markersize=7, label="うち個人")
ax.plot(years_t, hojin_b, "^--", color="#FF9800", linewidth=2, markersize=7, label="うち法人")
ax.set_ylabel("寄附金収入（十億円）")
ax.set_title("公益法人全体の寄附金収入額推移（H20～H24）", fontsize=14, fontweight="bold")
ax.legend()
for i, v in enumerate(total_b):
    ax.annotate(f"{v:.0f}", (i, v), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\n--- 寄附金収入総額（十億円）---")
for y, t, k, h in zip(years_t, total_b, kojin_b, hojin_b):
    print(f"  {y}: 計{t:.0f}, 個人{k:.0f}, 法人{h:.0f}")

**考察**: H23年度（2011年、東日本大震災年）に寄附金収入が急増（378十億円）。特に個人寄附が前年比2.2倍に。H24以降も震災前より高水準を維持。

## 4. 税額控除制度の効果分析

In [ ]:
rows = get_rows(wb_t, "３－４．税額控除制度導入前後の増加率の比較")
cats_34 = ["寄附金収入計", "個人寄附", "法人寄附"]
rate_all = [float(rows[6][9]), float(rows[7][9]), float(rows[8][9])]
rate_tc = [float(rows[6][10]), float(rows[7][10]), float(rows[8][10])]
rate_ntc = [float(rows[6][11]), float(rows[7][11]), float(rows[8][11])]

x = np.arange(len(cats_34))
w = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, [r*100 for r in rate_all], w, label="全体", color="#607D8B")
ax.bar(x, [r*100 for r in rate_tc], w, label="税額控除対象法人", color="#4CAF50")
ax.bar(x + w, [r*100 for r in rate_ntc], w, label="非対象法人", color="#FF7043")
ax.set_xticks(x)
ax.set_xticklabels(cats_34)
ax.set_ylabel("増加率（%）")
ax.set_title("税額控除制度 導入前後の寄附金増加率比較", fontsize=14, fontweight="bold")
ax.legend()
ax.axhline(y=0, color="black", linewidth=0.5)
for i in range(len(cats_34)):
    for j, (vals, offset) in enumerate([(rate_all, -w), (rate_tc, 0), (rate_ntc, w)]):
        ax.text(i + offset, vals[i]*100 + 2, f"{vals[i]*100:.0f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\n--- 増加率比較 ---")
for c, a, t, n in zip(cats_34, rate_all, rate_tc, rate_ntc):
    print(f"  {c}: 全体{a*100:.0f}%, 税控除{t*100:.0f}%, 非対象{n*100:.0f}%")

**考察**: 税額控除制度の導入（H23年度）後、税額控除対象法人の個人寄附は**+194%**と劇的に増加。非対象法人でも+92%の増加（震災効果含む）。法人寄附は税額控除対象法人で+60%、非対象で+7%と差が大きい。制度の効果が明確に表れている。

## 5. 寄附金の必要性と活動状況

In [ ]:
# 必要性
rows = get_rows(wb_s, "図16")
labels_q4 = [str(v).replace("１．", "").replace("２．", "") for v in rows[7][4:6]]
vals_q4 = [float(v) for v in rows[9][4:6]]

# 活動有無
rows6 = get_rows(wb_s, "図23")
labels_q6 = [str(v).replace("１．", "").replace("２．", "") for v in rows6[7][4:6]]
vals_q6 = [float(v) for v in rows6[9][4:6]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.pie(vals_q4, labels=labels_q4, autopct="%1.1f%%",
        colors=["#4CAF50", "#FF5722"], startangle=90, textprops={"fontsize": 12})
ax1.set_title("寄附金収入の必要性", fontsize=13, fontweight="bold")

ax2.pie(vals_q6, labels=labels_q6, autopct="%1.1f%%",
        colors=["#2196F3", "#9E9E9E"], startangle=90, textprops={"fontsize": 12})
ax2.set_title("寄附金を得るための活動の有無", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"必要: {vals_q4[0]:.1f}%, 不要: {vals_q4[1]:.1f}%")
print(f"活動あり: {vals_q6[0]:.1f}%, 活動なし: {vals_q6[1]:.1f}%")

**考察**: 寄附金収入が「必要」と回答した法人は約50%（推定）。しかし実際に寄附金獲得活動を「行っている」法人はさらに少ない。**必要性と行動のギャップ**が存在する。

## 6. 募金活動の手段

In [ ]:
rows = get_rows(wb_s, "図28")
method_labels = [str(v) for v in rows[7][4:] if v]
method_vals_all = [float(v) for v in rows[8][4:4+len(method_labels)]]
method_vals_kojin = [float(v) for v in rows[10][4:4+len(method_labels)]]
method_vals_hojin = [float(v) for v in rows[12][4:4+len(method_labels)]]

x = np.arange(len(method_labels))
w = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, method_vals_all, w, label="全体", color="#607D8B")
ax.bar(x, method_vals_kojin, w, label="個人向け", color="#2196F3")
ax.bar(x + w, method_vals_hojin, w, label="法人向け", color="#FF9800")
ax.set_xticks(x)
ax.set_xticklabels(method_labels, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("構成比（%）")
ax.set_title("寄附金募集の具体的手段（対象別比較）", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 7. 寄附金の使途

In [ ]:
rows = get_rows(wb_t, "５．寄附金の使途")
usage_labels = ["公益目的事業費", "収益事業等事業費", "管理費", "資産取得費"]
years_u = ["H20", "H21", "H22", "H23", "H24"]
usage_all = {}
for i, label in enumerate(usage_labels):
    row = rows[7 + i]
    usage_all[label] = [float(v)*100 for v in row[2:7]]

fig, ax = plt.subplots(figsize=(10, 5))
for i, (label, vals) in enumerate(usage_all.items()):
    ax.plot(years_u, vals, "o-", color=COLORS[i], linewidth=2, markersize=7, label=label)
ax.set_ylabel("充当割合（%）")
ax.set_title("寄附金の使途別充当割合の推移（公益法人全体）", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print("\n--- H24年度 充当割合 ---")
for label, vals in usage_all.items():
    print(f"  {label}: {vals[-1]:.1f}%")

**考察**: 寄附金の最大の使途は「公益目的事業費」（約24%）で、法人の本来の目的に充当されている。管理費への充当は約17%で、年々やや増加傾向。税額控除対象法人に限定すると公益目的事業費への充当率は約64%と顕著に高い。

## 8. 税額控除対象法人の状況

In [ ]:
rows = get_rows(wb_s, "図36")
tc_labels = [str(v).replace("１．", "").replace("２．", "").replace("３．", "") for v in rows[7][4:7] if v]
tc_vals = [float(v) for v in rows[9][4:7] if v is not None]

fig, ax = plt.subplots(figsize=(7, 7))
colors_tc = ["#4CAF50", "#FF5722", "#9E9E9E"]
wedges, texts, autotexts = ax.pie(tc_vals, labels=tc_labels, autopct="%1.1f%%",
                                   colors=colors_tc, startangle=90, textprops={"fontsize": 11})
ax.set_title("税額控除対象法人の該当状況", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

for l, v in zip(tc_labels, tc_vals):
    print(f"  {l}: {v:.1f}%")

**考察**: 税額控除対象法人はわずか8.6%（341法人）。大多数の公益法人は税額控除の恩恵を受けていない。PST要件を満たしていると回答した法人は230法人（6.3%）に留まる。

## 9. 総合考察

### 主要知見

1. **寄附金の二極化**: 過半数（約55%）の公益法人は寄附金収入がゼロ。一方、少数の大規模法人に寄附が集中
2. **税額控除制度の効果**: 制度導入後、対象法人の個人寄附は+194%と劇的に増加。制度による明確なインセンティブ効果が確認
3. **法人寄附が主力**: 寄附金総額の約7割は法人からの寄附。個人寄附の拡大余地は大きい
4. **活動ギャップ**: 寄附金の必要性を認識しつつも、募金活動を行っていない法人が多い
5. **PST要件の壁**: 税額控除対象法人は8.6%に留まる。小規模法人にとってPST要件のハードルが高い

### 政策的示唆

- **PST要件の緩和**: H28年度税制改正で公益目的事業費用1億円未満の法人に対する要件緩和が実施されたが、さらなる緩和が望まれる
- **デジタル化の推進**: ホームページ掲載が最も一般的な募金手段だが、オンライン決済やクラウドファンディングの活用促進が有効
- **情報発信支援**: 募金活動のノウハウや成功事例の共有により、「活動していない」法人の底上げが可能

### データの限界

- 自記式調査のため回答バイアスの可能性
- 回収率46.8%で、非回答法人の特性が不明
- 集計表のH20〜H24データとsyukeidata_r1のH26〜H30データで時期が異なる点に注意

In [ ]:
wb_s.close()
wb_t.close()
print("分析完了")